# Llama 3.2 1B — Memory-Safe Unsloth QLoRA Fine-Tuning

A clean replacement notebook for fine-tuning `meta-llama/Llama-3.2-1B-Instruct` on the counter-drone dataset.

Key fixes:

- installs Unsloth as one coherent dependency set
- uses a microbatch of **1**, not 4 long sequences at once
- preserves effective batch size with gradient accumulation
- uses prompt/completion data and completion-only loss
- measures the complete token-length distribution
- includes a one-batch VRAM smoke test and detailed CUDA memory reports
- uses the Llama chat template for both training and inference

Run the install cell once, restart the runtime if packages were already imported, then run top to bottom.

## 1. Install dependencies

In [15]:
# Do not separately upgrade TRL, PEFT, Accelerate, or bitsandbytes afterward.
%pip install -U -q unsloth datasets huggingface_hub

print("Install complete. Restart the runtime if these packages were already imported.")

Install complete. Restart the runtime if these packages were already imported.


## 2. Imports and environment checks

In [16]:
import os, sys, gc, math, random, inspect
from pathlib import Path

import numpy as np
import torch
from datasets import load_dataset
from huggingface_hub import login

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

assert torch.cuda.is_available(), "A CUDA GPU is required."
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU capacity: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")
print("BF16 supported:", torch.cuda.is_bf16_supported())

Python: 3.12.13
PyTorch: 2.11.0+cu128
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
GPU capacity: 95.0 GiB
BF16 supported: True


## 3. Hugging Face authentication

In [17]:
def get_secret(name):
    # Read a secret from Colab, Kaggle, or the environment.
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value:
            return value
    except Exception:
        pass
    return os.getenv(name)

HF_TOKEN = get_secret("HF_TOKEN")
assert HF_TOKEN, "Add HF_TOKEN to Colab/Kaggle secrets or the environment."
login(token=HF_TOKEN, add_to_git_credential=False)
print("Authenticated with Hugging Face.")

Authenticated with Hugging Face.


## 4. Configuration

In [24]:
MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
DATASET_NAME = "JamesResearch1216/threat-detection-responses-10k"
DATASET_SPLIT = "train"

# Start at 3072. Raise it only after inspecting the measured length distribution.
MAX_SEQ_LENGTH = 3072
VALIDATION_FRACTION = 0.02
MAX_TRAIN_EXAMPLES = None  # Set an integer for a quick smoke test.

LORA_RANK = 8
LORA_ALPHA = 16

OUTPUT_DIR = "./llama-drone-adapter"
HUB_MODEL_ID = "JamesResearch1216/llama-drone-adapter-v2"
PUSH_TO_HUB = False

NUM_TRAIN_EPOCHS = 1
MICRO_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 2e-4
PACKING = False

print("Effective batch size:", MICRO_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
print("Maximum sequence length:", MAX_SEQ_LENGTH)

Effective batch size: 128
Maximum sequence length: 3072


## 5. Load the 4-bit model and attach LoRA

In [19]:
from unsloth import FastLanguageModel, is_bfloat16_supported

compute_dtype = torch.bfloat16 if is_bfloat16_supported() else torch.float16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=compute_dtype,
    load_in_4bit=True,
    token=HF_TOKEN,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    # use_gradient_checkpointing="unsloth",
    use_gradient_checkpointing=False,
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
model.config.use_cache = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,}")
print(f"Represented parameters: {total:,}")
print(f"Trainable percentage: {100 * trainable / total:.4f}%")
print("Device:", next(model.parameters()).device)
print("Compute dtype:", compute_dtype)
print("Attention implementation:", getattr(model.config, "_attn_implementation", "not reported"))
print("Gradient checkpointing:", getattr(model, "is_gradient_checkpointing", "not reported"))

==((====))==  Unsloth 2026.7.5: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Trainable parameters: 5,636,096
Represented parameters: 780,077,056
Trainable percentage: 0.7225%
Device: cuda:0
Compute dtype: torch.bfloat16
Attention implementation: sdpa
Gradient checkpointing: False


## 6. CUDA memory utilities

In [20]:
def clear_gpu_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


def reset_peak_memory():
    clear_gpu_memory()
    torch.cuda.reset_peak_memory_stats()


def gpu_memory_report(label):
    torch.cuda.synchronize()
    gib = 1024**3
    print(f"\n[{label}]")
    print(f"Current allocated: {torch.cuda.memory_allocated() / gib:8.2f} GiB")
    print(f"Current reserved:  {torch.cuda.memory_reserved() / gib:8.2f} GiB")
    print(f"Peak allocated:    {torch.cuda.max_memory_allocated() / gib:8.2f} GiB")
    print(f"Peak reserved:     {torch.cuda.max_memory_reserved() / gib:8.2f} GiB")

reset_peak_memory()
gpu_memory_report("after model and LoRA setup")


[after model and LoRA setup]
Current allocated:     2.25 GiB
Current reserved:      2.85 GiB
Peak allocated:        2.25 GiB
Peak reserved:         2.85 GiB


## 7. System instruction

In [21]:
INSTRUCTION = """You are the AI assistant for an airport counter-drone threat-detection system. Operators, responders, and members of the public ask you what is happening — sometimes formally, sometimes anxiously, sometimes casually — and you give them a clear, faithful answer based on the current sensor readings.

THE SENSOR NETWORK
The site is covered by 23 sensors arranged in a ring around the airfield, indexed 0 through 22. Each sensor carries three independent detectors:
- Radio Frequency (RF): classifies radio emissions as friendly or threat.
- Audio: classifies sound as Mambo drone (threat), Bebop drone (threat), or Background noise (no threat).
- Visual: a camera score from 0 (no drone visible) to 1 (drone clearly visible).

A fusion model combines every sensor's readings into a single system-wide threat estimate. You receive that estimate at the top of each snapshot, then the per-sensor breakdown.

SENSOR GROUPINGS BY REGION
When the user asks about an area, use these named groups:
- Quadrants:
  - First quadrant: sensors 11-16
  - Second quadrant: sensors 5-11
  - Third quadrant: sensors 0-5
  - Fourth quadrant: sensors 16-22
- Hemispheres:
  - North: sensors 5-16
  - South: sensors 0-5 and 16-22
  - East: sensors 11-22
  - West: sensors 0-11

HOW TO ANSWER
- Ground every claim in the snapshot. NEVER invent numbers, sensors, or readings the snapshot does not show.
- Match the user's voice. A commander gets a clipped, direct answer; a worried bystander gets reassurance in plain words; a casual user gets a casual reply.
- For lay users, do NOT use technical words like "logit", "softmax", "modality", or "confidence vector". Use natural phrases.
- Do not dump the entire snapshot. Focus on what the user asked.
- When several sensors or detector types agree, say so.
- When asked about direction or area, use the named groupings above.
- Do not offer to display images, play audio, pull spectrograms, or take any action outside answering the question.
- Be honest about uncertainty. If a reading is borderline, around 50%, say so rather than overcommitting.
- Keep responses appropriately brief. One to four sentences is usually right.
"""

## 8. Load and format the dataset

The dataset is converted to TRL prompt/completion format. With `completion_only_loss=True`, the system prompt and sensor snapshot provide context, but loss is computed only on the target assistant response.

In [25]:
raw_dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT)
required = {"query", "snapshot_string", "response"}
missing = required - set(raw_dataset.column_names)
assert not missing, f"Missing columns: {sorted(missing)}; available: {raw_dataset.column_names}"

if MAX_TRAIN_EXAMPLES is not None:
    count = min(MAX_TRAIN_EXAMPLES, len(raw_dataset))
    raw_dataset = raw_dataset.shuffle(seed=SEED).select(range(count))


def format_example(example):
    user_content = (
        f"USER QUESTION:\n{example['query'].strip()}\n\n"
        f"CURRENT SENSOR READINGS:\n{example['snapshot_string'].strip()}"
    )
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": INSTRUCTION},
            {"role": "user", "content": user_content},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    completion = example["response"].strip()
    if tokenizer.eos_token and not completion.endswith(tokenizer.eos_token):
        completion += tokenizer.eos_token
    return {"prompt": prompt, "completion": completion}

formatted = raw_dataset.map(
    format_example,
    remove_columns=raw_dataset.column_names,
    desc="Formatting prompt/completion examples",
)
splits = formatted.train_test_split(test_size=VALIDATION_FRACTION, seed=SEED)
train_dataset = splits["train"]
eval_dataset = splits["test"]

print("Training examples:", len(train_dataset))
print("Validation examples:", len(eval_dataset))
print("\nPrompt preview:\n", train_dataset[0]["prompt"][:1200])
print("\nCompletion preview:\n", train_dataset[0]["completion"])

Resolving data files:   0%|          | 0/60 [00:00<?, ?it/s]

Training examples: 9408
Validation examples: 192

Prompt preview:
 <|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 23 Jul 2026

You are the AI assistant for an airport counter-drone threat-detection system. Operators, responders, and members of the public ask you what is happening — sometimes formally, sometimes anxiously, sometimes casually — and you give them a clear, faithful answer based on the current sensor readings.

THE SENSOR NETWORK
The site is covered by 23 sensors arranged in a ring around the airfield, indexed 0 through 22. Each sensor carries three independent detectors:
- Radio Frequency (RF): classifies radio emissions as friendly or threat.
- Audio: classifies sound as Mambo drone (threat), Bebop drone (threat), or Background noise (no threat).
- Visual: a camera score from 0 (no drone visible) to 1 (drone clearly visible).

A fusion model combines every sensor's readings into a single system-wide threat es

## 9. Measure all token lengths

In [26]:
def add_lengths(batch):
    prompt_tokens = tokenizer(batch["prompt"], add_special_tokens=False, truncation=False)["input_ids"]
    completion_tokens = tokenizer(batch["completion"], add_special_tokens=False, truncation=False)["input_ids"]
    p = [len(x) for x in prompt_tokens]
    c = [len(x) for x in completion_tokens]
    return {
        "prompt_tokens": p,
        "completion_tokens": c,
        "total_tokens": [a + b for a, b in zip(p, c)],
    }

length_dataset = train_dataset.map(add_lengths, batched=True, batch_size=128, desc="Token lengths")
lengths = np.asarray(length_dataset["total_tokens"])
for percentile in [0, 25, 50, 75, 90, 95, 99, 100]:
    print(f"p{percentile:>3}: {np.percentile(lengths, percentile):7.0f} tokens")

over = int((lengths > MAX_SEQ_LENGTH).sum())
print(f"\nOver {MAX_SEQ_LENGTH}: {over:,}/{len(lengths):,} ({100 * over / len(lengths):.2f}%)")
if over:
    print("These examples will be truncated. Raise MAX_SEQ_LENGTH only if their tail is important.")

p  0:    2704 tokens
p 25:    2740 tokens
p 50:    2754 tokens
p 75:    2768 tokens
p 90:    2781 tokens
p 95:    2790 tokens
p 99:    2808 tokens
p100:    2890 tokens

Over 3072: 0/9,408 (0.00%)


## 10. One-example forward/backward VRAM smoke test

In [27]:
RUN_MEMORY_SMOKE_TEST = True

if RUN_MEMORY_SMOKE_TEST:
    sample = train_dataset[0]
    full_text = sample["prompt"] + sample["completion"]
    batch = tokenizer(full_text, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)
    batch = {k: v.to(model.device) for k, v in batch.items()}
    batch["labels"] = batch["input_ids"].clone()

    prompt_ids = tokenizer(
        sample["prompt"], return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH
    )["input_ids"]
    prompt_length = min(prompt_ids.shape[1], batch["labels"].shape[1])
    batch["labels"][:, :prompt_length] = -100

    reset_peak_memory()
    model.train()
    output = model(**batch)
    output.loss.backward()
    gpu_memory_report("single-example forward/backward")
    print("Smoke-test loss:", float(output.loss.detach().cpu()))

    model.zero_grad(set_to_none=True)
    del output, batch, prompt_ids
    clear_gpu_memory()


[single-example forward/backward]
Current allocated:     2.31 GiB
Current reserved:      6.60 GiB
Peak allocated:        6.48 GiB
Peak reserved:         6.60 GiB
Smoke-test loss: 2.0457029342651367


## 11. Configure SFTTrainer

In [28]:
import math
import torch

gpu_major, gpu_minor = torch.cuda.get_device_capability()
SUPPORTS_TF32 = gpu_major >= 8

steps_per_epoch = math.ceil(
    len(train_dataset)
    / (
        MICRO_BATCH_SIZE
        * GRADIENT_ACCUMULATION_STEPS
    )
)

TOTAL_TRAINING_STEPS = steps_per_epoch * NUM_TRAIN_EPOCHS
WARMUP_STEPS = max(1, int(0.03 * TOTAL_TRAINING_STEPS))

print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", f"{gpu_major}.{gpu_minor}")
print("BF16:", is_bfloat16_supported())
print("FP16:", not is_bfloat16_supported())
print("TF32:", SUPPORTS_TF32)
print("Estimated optimizer steps:", TOTAL_TRAINING_STEPS)
print("Warmup steps:", WARMUP_STEPS)

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Compute capability: 12.0
BF16: True
FP16: False
TF32: True
Estimated optimizer steps: 74
Warmup steps: 2


In [29]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=MICRO_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,

    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    lr_scheduler_type="linear",
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=0.3,

    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    tf32=SUPPORTS_TF32,

    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
    packing=PACKING,

    logging_strategy="steps",
    logging_steps=10,
    logging_first_step=True,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    report_to="none",
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0,
    skip_memory_metrics=False,
    include_num_input_tokens_seen=True,

    push_to_hub=PUSH_TO_HUB,
    hub_model_id=HUB_MODEL_ID if PUSH_TO_HUB else None,
    hub_strategy="every_save" if PUSH_TO_HUB else "end",
    hub_token=HF_TOKEN if PUSH_TO_HUB else None,
)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_dataset,
    "eval_dataset": eval_dataset,
}

# Supports both current TRL (`processing_class`) and older releases (`tokenizer`).
params = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)
print("Trainer constructed.")
print("Microbatch:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Effective batch:", training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)

Unsloth: We found double BOS tokens - we shall remove one automatically.
Unsloth: We found double BOS tokens - we shall remove one automatically.
🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Trainer constructed.
Microbatch: 8
Gradient accumulation: 16
Effective batch: 128


## 12. Train

In [30]:
reset_peak_memory()
gpu_memory_report("before training")

train_result = trainer.train()

gpu_memory_report("after training")
print("\nTraining metrics")
for key, value in train_result.metrics.items():
    print(f"{key}: {value}")
trainer.save_state()


[before training]
Current allocated:     2.26 GiB
Current reserved:      2.85 GiB
Peak allocated:        2.26 GiB
Peak reserved:         2.85 GiB


/usr/local/lib/python3.12/dist-packages/unsloth/models/_utils.py:371: UserWarning: Unsloth: detected a manual forward/backward run before trainer.train(); reset the torch.compile graph cache it poisoned so training starts clean. To avoid this, run any pre-train probe under `with torch.no_grad():`.
  warnings.warn(
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,408 | Num Epochs = 1 | Total steps = 74
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 16 x 1) = 128
 "-____-"     Trainable parameters = 5,636,096 of 1,241,450,496 (0.45% trained)


Step,Training Loss,Validation Loss,Input Tokens Seen
74,0.930121,0.884950,25917512


Unsloth: Restored added_tokens_decoder metadata in ./llama-drone-adapter/checkpoint-74/tokenizer_config.json.



[after training]
Current allocated:     2.27 GiB
Current reserved:      2.89 GiB
Peak allocated:        6.57 GiB
Peak reserved:        16.59 GiB

Training metrics
train_runtime: 858.7381
train_samples_per_second: 10.956
train_steps_per_second: 0.086
total_flos: 1.5220559772735898e+17
train_loss: 1.113160262236724
init_mem_cpu_alloc_delta: 8192
init_mem_gpu_alloc_delta: 0
init_mem_cpu_peaked_delta: 0
init_mem_gpu_peaked_delta: 0
train_mem_cpu_alloc_delta: 477089792
train_mem_gpu_alloc_delta: 11554304
train_mem_cpu_peaked_delta: 4096
train_mem_gpu_peaked_delta: 4610346496
before_init_mem_cpu: 3300503552
before_init_mem_gpu: 2428613632
epoch: 1.0
num_input_tokens_seen: 25917512


## 13. Evaluate and save the adapter

In [31]:
original_report_to = trainer.args.report_to
trainer.args.report_to = []

eval_metrics = trainer.evaluate()

trainer.args.report_to = original_report_to

print("Validation metrics")
for key, value in eval_metrics.items():
    print(f"{key}: {value}")
if "eval_loss" in eval_metrics and eval_metrics["eval_loss"] < 20:
    print("Perplexity:", math.exp(eval_metrics["eval_loss"]))

save_path = Path(OUTPUT_DIR) / "final_adapter"
save_path.mkdir(parents=True, exist_ok=True)
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print("\nSaved adapter to:", save_path.resolve())
for path in sorted(save_path.iterdir()):
    if path.is_file():
        print(f"  {path.name:35s} {path.stat().st_size / 1024**2:8.2f} MiB")

if PUSH_TO_HUB:
    trainer.push_to_hub(commit_message="Upload final QLoRA adapter")

RuntimeError: on_train_begin must be called before on_evaluate

## 14. Inference sanity check

In [32]:
TEST_QUERY = "Report if there is an audio or RF alert on the southwest border."
TEST_SNAPSHOT = """Sensor 0 data:
Radio Frequency (RF) Detection: 74.23% no drone threat
Audio Detection: 99.73% no drone threat
Visual Detection: 100.00% no drone threat

Sensor 16 data:
Radio Frequency (RF) Detection: 94.87% drone threat
Audio Detection: 100.00% drone threat
Drone Type:
Mambo (drone): 100.00%
Bebop (drone): 0.00%
Background (no drone): 0.00%
Visual Detection: 100.00% drone threat
"""

FastLanguageModel.for_inference(model)
model.config.use_cache = True
messages = [
    {"role": "system", "content": INSTRUCTION},
    {
        "role": "user",
        "content": f"USER QUESTION:\n{TEST_QUERY}\n\nCURRENT SENSOR READINGS:\n{TEST_SNAPSHOT}",
    },
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)

with torch.inference_mode():
    generated = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = generated[0, inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


There is a high alert on the southwest border, specifically around sensors 0 and 16. Both the radio and audio detections are indicating a drone threat, with sensor 16 showing a 94.87% chance of a drone threat.


## 15. Package version report

In [ ]:
import importlib.metadata as metadata

for package in [
    "unsloth", "unsloth_zoo", "torch", "transformers", "trl",
    "peft", "accelerate", "bitsandbytes", "datasets",
]:
    try:
        print(f"{package:15s} {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package:15s} not installed")

## Memory interpretation

- **Peak allocated** is the most useful measure of memory occupied by live tensors.
- **Peak reserved** can be larger because PyTorch caches CUDA blocks.
- A 1B 4-bit QLoRA model should not need approximately 70 GiB with microbatch 1 under a working optimized attention path.
- Keep `per_device_train_batch_size=1` for these long samples. Increase `gradient_accumulation_steps` to change effective batch size.
- Increase `MAX_SEQ_LENGTH` only after checking the token percentiles and truncation rate.

In [ ]:
from pathlib import Path
import os
import shutil

from huggingface_hub import HfApi, login

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
HF_REPO_ID = "JamesResearch1216/llama-drone-1b"

# Colab's normal writable workspace.
EXPORT_ROOT = Path("/content/llama_drone_release")

LORA_DIR = EXPORT_ROOT / "lora"
GGUF_DIR = EXPORT_ROOT / "gguf"

# Some Unsloth versions append "_gguf" to the requested directory.
UNSLOTH_GGUF_DIR = Path(str(GGUF_DIR) + "_gguf")

GGUF_QUANTIZATION = "q4_k_m"
PRIVATE_REPO = False

# Set True to force GGUF regeneration even when one already exists.
FORCE_GGUF_REEXPORT = False

EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
LORA_DIR.mkdir(parents=True, exist_ok=True)
GGUF_DIR.mkdir(parents=True, exist_ok=True)

assert HF_TOKEN, "HF_TOKEN is missing."
login(token=HF_TOKEN, add_to_git_credential=False)

# ------------------------------------------------------------------
# 1. Save the LoRA adapter and tokenizer
# ------------------------------------------------------------------
model.save_pretrained(str(LORA_DIR))
tokenizer.save_pretrained(str(LORA_DIR))

print("Saved LoRA adapter:")
for path in sorted(LORA_DIR.iterdir()):
    if path.is_file():
        print(
            f"  {path.name}: "
            f"{path.stat().st_size / 1024**2:.2f} MiB"
        )

# ------------------------------------------------------------------
# 2. Check whether a GGUF already exists
# ------------------------------------------------------------------
existing_gguf_files = []

for candidate_dir in [GGUF_DIR, UNSLOTH_GGUF_DIR]:
    if candidate_dir.exists():
        existing_gguf_files.extend(candidate_dir.rglob("*.gguf"))

existing_gguf_files = sorted(set(existing_gguf_files))

# ------------------------------------------------------------------
# 3. Export merged GGUF only when needed
# ------------------------------------------------------------------
if existing_gguf_files and not FORCE_GGUF_REEXPORT:
    print("\nExisting GGUF found; skipping conversion.")

    for path in existing_gguf_files:
        print(
            f"  {path}: "
            f"{path.stat().st_size / 1024**3:.2f} GiB"
        )
else:
    if FORCE_GGUF_REEXPORT:
        print("\nRemoving old GGUF files before re-export.")

        for directory in [GGUF_DIR, UNSLOTH_GGUF_DIR]:
            if directory.exists():
                shutil.rmtree(directory)

        GGUF_DIR.mkdir(parents=True, exist_ok=True)

    print("\nExporting merged GGUF...")

    model.save_pretrained_gguf(
        save_directory=str(GGUF_DIR),
        tokenizer=tokenizer,
        quantization_method=GGUF_QUANTIZATION,
    )

# ------------------------------------------------------------------
# 4. Locate Unsloth's actual output
# ------------------------------------------------------------------
generated_files = []

for candidate_dir in [GGUF_DIR, UNSLOTH_GGUF_DIR]:
    if candidate_dir.exists():
        generated_files.extend(
            path
            for path in candidate_dir.rglob("*")
            if path.is_file()
        )

if not generated_files:
    raise FileNotFoundError(
        "Unsloth finished, but no GGUF output files were found in:\n"
        f"  {GGUF_DIR}\n"
        f"  {UNSLOTH_GGUF_DIR}"
    )

# ------------------------------------------------------------------
# 5. Normalize everything into /content/llama_drone_release/gguf
# ------------------------------------------------------------------
GGUF_DIR.mkdir(parents=True, exist_ok=True)

for source_path in generated_files:
    # File is already directly inside the final directory.
    if source_path.parent == GGUF_DIR:
        continue

    destination_path = GGUF_DIR / source_path.name

    if destination_path.exists():
        if destination_path.is_dir():
            shutil.rmtree(destination_path)
        else:
            destination_path.unlink()

    shutil.move(
        str(source_path),
        str(destination_path),
    )

# Remove Unsloth's extra output directory after moving its files.
if UNSLOTH_GGUF_DIR.exists():
    shutil.rmtree(UNSLOTH_GGUF_DIR)

gguf_files = sorted(GGUF_DIR.glob("*.gguf"))

if not gguf_files:
    raise FileNotFoundError(
        f"No GGUF files were found in final directory: {GGUF_DIR}"
    )

print("\nFinal GGUF files:")
for path in gguf_files:
    print(
        f"  {path.name}: "
        f"{path.stat().st_size / 1024**3:.2f} GiB"
    )

# Prefer the requested quantization.
preferred_gguf_files = [
    path
    for path in gguf_files
    if GGUF_QUANTIZATION.lower() in path.name.lower()
]

main_gguf = (
    preferred_gguf_files[0]
    if preferred_gguf_files
    else gguf_files[0]
)

print("\nSelected deployment GGUF:", main_gguf.name)

# ------------------------------------------------------------------
# 6. Write an explicit Ollama Modelfile
# ------------------------------------------------------------------
modelfile_path = GGUF_DIR / "Modelfile"

modelfile_text = f'''FROM ./{main_gguf.name}

PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
PARAMETER num_predict 256

PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|end_of_text|>"

SYSTEM """
{INSTRUCTION.strip()}
"""
'''

modelfile_path.write_text(
    modelfile_text,
    encoding="utf-8",
)

print("Wrote Modelfile:", modelfile_path)

# ------------------------------------------------------------------
# 7. Save the system instruction separately
# ------------------------------------------------------------------
instruction_path = EXPORT_ROOT / "system_instruction.txt"

instruction_path.write_text(
    INSTRUCTION.strip() + "\n",
    encoding="utf-8",
)

# ------------------------------------------------------------------
# 8. Write repository README
# ------------------------------------------------------------------
readme = f"""---
base_model: meta-llama/Llama-3.2-1B-Instruct
library_name: peft
pipeline_tag: text-generation
tags:
- unsloth
- lora
- gguf
- ollama
- llama
---

# Llama Drone 1B

This repository contains a counter-drone assistant fine-tuned from
`meta-llama/Llama-3.2-1B-Instruct`.

## Repository contents

- `lora/` — LoRA adapter and tokenizer files
- `gguf/{main_gguf.name}` — merged `{GGUF_QUANTIZATION}` GGUF model
- `gguf/Modelfile` — Ollama deployment configuration
- `system_instruction.txt` — deployment system instruction

The GGUF already contains the merged LoRA changes. Do not apply the
separate LoRA adapter to the GGUF again.

## Ollama usage

Download the repository, enter the `gguf` directory, and run:

```bash
ollama create llama-drone -f Modelfile
ollama run llama-drone
"""